In [1]:
import cadquery as cq
from cadquery import exporters
import math
from fractions import Fraction
from jupyter_cadquery import *

from jupyter_cadquery.replay import replay, enable_replay, disable_replay

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

versions()

Overwriting auto display for cadquery Workplane and Shape

Versions:
- jupyter_cadquery  4.0.2
- cad_viewer_widget 3.0.2
- open cascade      7.7.2.1



In [2]:
cv = open_viewer("Rafter", cad_width=800, height=600)
set_default_viewer("Rafter")

In [3]:
# Rafter parameters
rafter_thickness = 3.0
rafter_depth = 6.0

historical_pitches = {
    "5/6": Fraction(5, 6),
    "4/5": Fraction(4, 5),
    "7/9": Fraction(7, 9),
    "2/3": Fraction(2, 3),
    "5/8": Fraction(5, 8),
    "3/5": Fraction(3, 5),
    "7/12": Fraction(7, 12),
    "4/7": Fraction(4, 7),
    "5/9": Fraction(5, 9),
    # Special "right-angle roof" mentioned separately
    "17/24": Fraction(17, 24),
}


def historical_pitch_angle(proportion):
    p = float(proportion)

    return math.degrees(
        math.acos(1.0 / (2.0 * p))
    )


def ridge_angle(proportion):
    slope = historical_pitch_angle(proportion)
    return 180.0 - 2.0 * slope


for name, proportion in historical_pitches.items():
    slope = historical_pitch_angle(proportion)
    ridge = ridge_angle(proportion)

    lap = (
        rafter_depth
        / math.sin(math.radians(ridge))
    )

    print(
        f"{name:>5}  "
        f"slope={slope:6.2f}°  "
        f"ridge={ridge:6.2f}°  "
        f"lap={lap:6.3f}\""
    )

  5/6  slope= 53.13°  ridge= 73.74°  lap= 6.250"
  4/5  slope= 51.32°  ridge= 77.36°  lap= 6.149"
  7/9  slope= 49.99°  ridge= 80.01°  lap= 6.092"
  2/3  slope= 41.41°  ridge= 97.18°  lap= 6.047"
  5/8  slope= 36.87°  ridge=106.26°  lap= 6.250"
  3/5  slope= 33.56°  ridge=112.89°  lap= 6.513"
 7/12  slope= 31.00°  ridge=117.99°  lap= 6.795"
  4/7  slope= 28.96°  ridge=122.09°  lap= 7.082"
  5/9  slope= 25.84°  ridge=128.32°  lap= 7.647"
17/24  slope= 45.10°  ridge= 89.80°  lap= 6.000"


In [4]:
# Roof parameters
roof_pitch = Fraction(2, 3)
attic_floor_joist_length = 256.0
slope = historical_pitch_angle(roof_pitch)
ridge = ridge_angle(roof_pitch)
false_plate_top_z = 0

# Rafter calculations
rafter_length = (
    attic_floor_joist_length
    * float(roof_pitch)
)

# Half span
half_span = attic_floor_joist_length / 2

# Roof angle
roof_angle = math.degrees(
    math.acos(
        half_span / rafter_length
    )
)

# Roof angle radians
a = math.radians(roof_angle)
ridge_deg = 180.0 - 2.0 * roof_angle

# Rafter rise
rafter_rise = math.sqrt(
    rafter_length**2
    - half_span**2
)

# Rafter run
rafter_run = half_span
ridge_z = rafter_rise

# Centerline distance from ridge to false-plate bearing
bearing_x = rafter_length

print(f"roof angle: {roof_angle:.2f}°")
print(f"rafter rise: {rafter_rise:.3f}\"")
print(f"rafter run: {rafter_run:.3f}\"")

roof angle: 41.41°
rafter rise: 112.885"
rafter run: 128.000"


In [5]:
# Collar
collar_height = 6.0
collar_thickness = rafter_thickness

# World elevation of collar centerline
collar_center_z = ridge_z - 36.0

# Half-dovetail seat
seat_depth = 0.5
cut_margin = 0.25

# Half-lap through Y
collar_lap_depth = collar_thickness / 2
collar_clearance = 0.02

In [6]:
s = math.sin(a)
c = math.cos(a)
right_normal_x = s
right_normal_z = c
left_normal_x = -math.sin(a)
left_normal_z =  math.cos(a)
half_crossing = rafter_depth / (2 * s)

In [7]:
def intersect_lines_2d(p, d, q, e):
    """
    Solve:
        p + t*d = q + u*e

    where p,q are (x,z) points and d,e are (dx,dz) directions.
    Returns the intersection point (x,z).
    """
    px, pz = p
    dx, dz = d
    qx, qz = q
    ex, ez = e

    den = dx * ez - dz * ex
    if abs(den) < 1e-9:
        raise ValueError("Lines are parallel or nearly parallel")

    rx = qx - px
    rz = qz - pz

    t = (rx * ez - rz * ex) / den

    return (
        px + t * dx,
        pz + t * dz,
    )

In [8]:
def right_center_x_at_z(z):
    return (ridge_z - z) / math.tan(a)

def right_inner_x_at_z(z):
    return right_center_x_at_z(z) - half_crossing

def right_outer_x_at_z(z):
    return right_center_x_at_z(z) + half_crossing


def left_center_x_at_z(z):
    return -(ridge_z - z) / math.tan(a)

def left_inner_x_at_z(z):
    return left_center_x_at_z(z) + half_crossing

def left_outer_x_at_z(z):
    return left_center_x_at_z(z) - half_crossing

# Collar z
collar_z0 = collar_center_z - collar_height / 2
collar_z1 = collar_center_z + collar_height / 2
seat_z = collar_z1 - seat_depth

# Seat run
seat_run = abs(
    right_outer_x_at_z(collar_z1)
    - right_inner_x_at_z(collar_z1)
)

# Seat angle
seat_angle = math.degrees(
    math.atan2(seat_depth, seat_run)
)

print(f"seat depth: {seat_depth:.3f}\"")
print(f"seat run: {seat_run:.3f}\"")
print(f"derived taper: {seat_angle:.2f}°")

seat depth: 0.500"
seat run: 9.071"
derived taper: 3.15°


In [9]:
def point_on_line_at_z(p, d, target_z):
    px, pz = p
    dx, dz = d

    if abs(dz) < 1e-9:
        raise ValueError("Line has no Z component")

    t = (target_z - pz) / dz

    return (
        px + t * dx,
        target_z,
    )

In [10]:
def xz_profile_solid(points, y0, y1):
    y_center = (y0 + y1) / 2
    y_thickness = y1 - y0

    return (
        cq.Workplane("XZ")
        .polyline(points)
        .close()
        .extrude(y_thickness / 2, both=True)
        .translate((0, y_center, 0))
        .val()
    )

In [11]:
right_rafter_dir = (
    math.cos(a),
    -math.sin(a)
)

right_normal_inward = (
    -math.sin(a),
    math.cos(a)
)

right_seat_dir = (
    math.cos(math.radians(seat_angle)),
    -math.sin(math.radians(seat_angle)),
)

right_shoulder_dir = (
    -math.sin(a),
    math.cos(a),
)

right_inner_top = (
    right_inner_x_at_z(collar_z1),
    collar_z1,
)

right_inner_bottom = (
    right_inner_x_at_z(collar_z0),
    collar_z0,
)

# Bottom of the small seat cut on the inside rafter face
right_inner_seat = (
    right_inner_x_at_z(seat_z),
    seat_z,
)

right_outer_top = (
    right_outer_x_at_z(collar_z1),
    collar_z1,
)

right_outer_bottom = (
    right_outer_x_at_z(collar_z0),
    collar_z0,
)

right_knee = point_on_line_at_z(
    right_outer_bottom,
    right_shoulder_dir,
    collar_z1,
)

right_taper_run = abs(
    right_outer_top[0] - right_inner_seat[0]
)

right_taper_angle = math.degrees(
    math.atan2(
        seat_depth,
        right_taper_run,
    )
)

print("right taper:", right_taper_angle)

right_tail_profile = [
    right_outer_bottom,
    right_knee,
    right_inner_seat,
    right_inner_top,
]

right taper: 3.3648016246379013


In [12]:
left_rafter_dir = (
    -math.cos(a),
    -math.sin(a)
)

left_normal_inward = (
    math.sin(a),
    math.cos(a)
)

left_seat_dir = (
    -math.cos(math.radians(seat_angle)),
    -math.sin(math.radians(seat_angle)),
)

left_shoulder_dir = (
    math.sin(a),
    math.cos(a),
)

left_inner_top = (
    left_inner_x_at_z(collar_z1),
    collar_z1,
)

left_inner_seat = (
    left_inner_x_at_z(seat_z),
    seat_z,
)

left_outer_top = (
    left_outer_x_at_z(collar_z1),
    collar_z1,
)

left_outer_bottom = (
    left_outer_x_at_z(collar_z0),
    collar_z0,
)

left_knee = point_on_line_at_z(
    left_outer_bottom,
    left_shoulder_dir,
    collar_z1,
)

left_inner_bottom = (
    left_inner_x_at_z(collar_z0),
    collar_z0,
)

left_taper_angle = math.degrees(
    math.atan2(
        left_knee[1] - left_inner_seat[1],
        abs(left_knee[0] - left_inner_seat[0]),
    )
)

print("left taper angle:", left_taper_angle)

left_tail_profile = [
    left_outer_bottom,
    left_inner_top,
    left_inner_seat,
    left_knee,
]

left taper angle: 2.857823606251247


In [13]:
def ridge_lap_length(member_depth, slope_deg):
    return (
        member_depth
        / math.sin(math.radians(2 * slope_deg))
    )

In [14]:
def ridge_lap_depth(member_thickness):
    return member_thickness / 2

In [15]:
lap_length = ridge_lap_length(
    rafter_depth,
    slope,
)

lap_depth = rafter_thickness / 2

lap_start = -lap_length / 2
lap_end   =  lap_length / 2

print(f"roof angle: {roof_angle:.2f}°")
print(f"lap length: {lap_length:.3f}\"")
print(f"lap depth: {lap_depth:.3f}\"")

roof angle: 41.41°
lap length: 6.047"
lap depth: 1.500"


In [16]:
alpha = math.radians(ridge_deg)

s = math.sin(alpha)
c = math.cos(alpha)

if s <= 0:
    raise ValueError(f"Invalid ridge angle: {ridge_deg}")

z_bottom = -rafter_depth / 2
z_top = rafter_depth / 2

right_lower_x_bottom = (
    bearing_x
    + z_bottom / math.tan(a)
)

right_lower_x_top = (
    bearing_x
    + z_top / math.tan(a)
)

left_lower_x_bottom = (
    bearing_x
    - z_bottom / math.tan(a)
)

left_lower_x_top = (
    bearing_x
    - z_top / math.tan(a)
)

In [17]:
def ridge_lap_coords(hand):
    """
    hand = +1 : right rafter
    hand = -1 : left rafter
    """

    def base_x(z):
        return -hand * z * c

    end_bottom_x = (
        base_x(z_bottom) - rafter_depth / 2
    ) / s

    end_top_x = (
        base_x(z_top) - rafter_depth / 2
    ) / s

    shoulder_bottom_x = (
        base_x(z_bottom) + rafter_depth / 2
    ) / s

    shoulder_top_x = (
        base_x(z_top) + rafter_depth / 2
    ) / s

    return (
        end_bottom_x,
        end_top_x,
        shoulder_bottom_x,
        shoulder_top_x,
    )

In [18]:
(
    right_end_bottom_x,
    right_end_top_x,
    right_shoulder_bottom_x,
    right_shoulder_top_x,
) = ridge_lap_coords(+1)


(
    left_end_bottom_x,
    left_end_top_x,
    left_shoulder_bottom_x,
    left_shoulder_top_x,
) = ridge_lap_coords(-1)

In [19]:
def local_prism(x0, x1, y0, y1, z0, z1):
    return cq.Solid.makeBox(
        x1 - x0,
        y1 - y0,
        z1 - z0,
        cq.Vector(x0, y0, z0),
    )

In [20]:
def place_right_rafter(shape):
    return (
        shape
        .rotate(
            (0, 0, 0),
            (0, 1, 0),
            roof_angle,
        )
        .translate((0, 0, ridge_z))
    )


def place_left_rafter(shape):
    return (
        shape
        .rotate(
            (0, 0, 0),
            (0, 1, 0),
            180.0 - roof_angle,
        )
        .translate((0, 0, ridge_z))
    )

In [21]:
right_rafter_body_local = (
    cq.Workplane("XZ")
    .moveTo(
        right_end_bottom_x,
        z_bottom,
    )
    .lineTo(
        right_end_top_x,
        z_top,
    )
    .lineTo(
        right_lower_x_top,
        z_top,
    )
    .lineTo(
        right_lower_x_bottom,
        z_bottom,
    )
    .close()
    .extrude(
        rafter_thickness / 2,
        both=True,
    )
    .val()
)

In [22]:
right_lap_profile = (
    cq.Workplane("XZ")
    .moveTo(
        right_end_bottom_x,
        z_bottom,
    )
    .lineTo(
        right_end_top_x,
        z_top,
    )
    .lineTo(
        right_shoulder_top_x,
        z_top,
    )
    .lineTo(
        right_shoulder_bottom_x,
        z_bottom,
    )
    .close()
)

right_ridge_cutter_local = (
    right_lap_profile
    .extrude(
        rafter_thickness / 4,
        both=True,
    )
    .translate(
        (0, rafter_thickness / 4, 0)
    )
    .val()
)

right_rafter_local = (
    right_rafter_body_local
    .cut(right_ridge_cutter_local)
    .clean()
)

In [23]:
left_rafter_body_local = (
    cq.Workplane("XZ")
    .moveTo(
        left_end_bottom_x,
        z_bottom,
    )
    .lineTo(
        left_end_top_x,
        z_top,
    )
    .lineTo(
        left_lower_x_top,
        z_top,
    )
    .lineTo(
        left_lower_x_bottom,
        z_bottom,
    )
    .close()
    .extrude(
        rafter_thickness / 2,
        both=True,
    )
    .val()
)

In [24]:
left_lap_profile = (
    cq.Workplane("XZ")
    .moveTo(
        left_end_bottom_x,
        z_bottom,
    )
    .lineTo(
        left_end_top_x,
        z_top,
    )
    .lineTo(
        left_shoulder_top_x,
        z_top,
    )
    .lineTo(
        left_shoulder_bottom_x,
        z_bottom,
    )
    .close()
)

left_ridge_cutter_local = (
    left_lap_profile
    .extrude(
        rafter_thickness / 4,
        both=True,
    )
    .translate(
        (0, -rafter_thickness / 4, 0)
    )
    .val()
)

left_rafter_local = (
    left_rafter_body_local
    .cut(left_ridge_cutter_local)
    .clean()
)

In [25]:
right_rafter = place_right_rafter(
    right_rafter_local
)

left_rafter = place_left_rafter(
    left_rafter_local
)

In [26]:
show_object(right_rafter, name="right")
show_object(left_rafter, name="left")

+
++


In [27]:
left_body_x = left_inner_x_at_z(collar_center_z)
right_body_x = right_inner_x_at_z(collar_center_z)

collar_profile = (
    cq.Workplane("XZ")

    # left outside bottom
    .moveTo(*left_outer_bottom)

    # outside face of left rafter
    .lineTo(*left_outer_top)

    # dovetail taper down to seat
    .lineTo(*left_inner_seat)

    # small seat cut along inside rafter face
    .lineTo(*left_inner_top)

    # untouched top of collar body
    .lineTo(*right_inner_top)

    # seat cut down along inside face
    .lineTo(*right_inner_seat)

    # dovetail taper back up to full blank height
    .lineTo(*right_outer_top)

    # outside rafter face down to collar bottom
    .lineTo(*right_outer_bottom)

    # original collar bottom remains level
    .lineTo(*left_outer_bottom)

    .close()
)

collar_full = (
    collar_profile
    .extrude(
        collar_thickness / 2,
        both=True,
    )
    .val()
)

In [28]:
# Cut parameters
cut_margin = 2.0
y_overlap = 0.02

# remove +Y half at the two lap ends
cut_y0 = -y_overlap
cut_y1 = collar_thickness / 2 + y_overlap

left_lap_x0 = min(
    left_outer_bottom[0],
    left_outer_top[0],
    left_inner_seat[0],
    left_inner_top[0],
)

left_lap_x1 = max(
    left_outer_bottom[0],
    left_outer_top[0],
    left_inner_seat[0],
    left_inner_top[0],
)

right_lap_x0 = min(
    right_outer_bottom[0],
    right_outer_top[0],
    right_inner_seat[0],
    right_inner_top[0],
)

right_lap_x1 = max(
    right_outer_bottom[0],
    right_outer_top[0],
    right_inner_seat[0],
    right_inner_top[0],
)

In [29]:
cut_margin = 2.0
y_overlap = 0.02

right_cut_profile = (
    cq.Workplane("XZ")

    # bottom of inside rafter face
    .moveTo(*right_inner_bottom)

    # top of inside rafter face
    .lineTo(*right_inner_top)

    # extend well beyond the collar end
    .lineTo(
        right_outer_top[0] + cut_margin,
        collar_z1 + cut_margin,
    )

    .lineTo(
        right_outer_bottom[0] + cut_margin,
        collar_z0 - cut_margin,
    )

    .close()
)

left_cut_profile = (
    cq.Workplane("XZ")

    .moveTo(*left_inner_bottom)
    .lineTo(*left_inner_top)

    .lineTo(
        left_outer_top[0] - cut_margin,
        collar_z1 + cut_margin,
    )

    .lineTo(
        left_outer_bottom[0] - cut_margin,
        collar_z0 - cut_margin,
    )

    .close()
)

In [30]:
left_half_lap_cutter = (
    left_cut_profile
    .extrude(
        collar_thickness / 4 + y_overlap / 2,
        both=True,
    )
    .translate(
        (
            0,
            collar_thickness / 4,
            0,
        )
    )
    .val()
)

right_half_lap_cutter = (
    right_cut_profile
    .extrude(
        collar_thickness / 4 + y_overlap / 2,
        both=True,
    )
    .translate(
        (
            0,
            collar_thickness / 4,
            0,
        )
    )
    .val()
)

In [31]:
collar = (
    collar_full
    .cut(left_half_lap_cutter)
    .cut(right_half_lap_cutter)
    .clean()
)

In [32]:
show_object(collar, name="collar")

+++
